# OFPP pseudopotentials + density optimization

Minimal DFTpy workflows for FCC Al, all using density optimization (TFvW + LDA):

1. **PP on disk** — explicit `PP_list`
2. **PP not on disk** — omit `PP_list`; OFPP auto-resolves
3. **Choose a PP family** — `families=...` on `LocalPseudo`
4. **Config API** — `[OFPP] auto = true` via `DefaultOption` / `ConfigParser`

Optional setup for Google Colab:

```bash
!python -m pip install "git+https://github.com/Quantum-MultiScale/DFTpy.git@dev"
!git clone --depth 1 https://github.com/Quantum-MultiScale/DFTpy.git
%cd DFTpy/examples/notebooks
```

In [1]:
from pathlib import Path

from ase.build import bulk

from dftpy.config import DefaultOption, OptionFormat
from dftpy.field import DirectField
from dftpy.functional import Functional, TotalFunctional
from dftpy.functional.pseudo import LocalPseudo
from dftpy.grid import DirectGrid
from dftpy.ions import Ions
from dftpy.interface import ConfigParser
from dftpy.math_utils import ecut2nr
from dftpy.optimization import Optimization

DATA = Path("../DATA").resolve()


def make_al_system(ecut=35):
    ions = Ions.from_ase(bulk("Al", "fcc", a=4.05, cubic=True))
    grid = DirectGrid(lattice=ions.cell, nr=ecut2nr(ecut=ecut, lattice=ions.cell))
    return ions, grid


def guess_density(ions, grid):
    """Uniform guess; call *after* LocalPseudo so Zval is set from the PP."""
    rho = DirectField(grid=grid)
    rho[:] = ions.get_ncharges() / ions.cell.volume
    return rho


def optimize_density(pseudo, ions, rho0):
    evaluator = TotalFunctional(
        KE=Functional(type="KEDF", name="TFvW"),
        XC=Functional(type="XC", name="LDA"),
        HARTREE=Functional(type="HARTREE"),
        PSEUDO=pseudo,
    )
    opt = Optimization(
        EnergyEvaluator=evaluator,
        optimization_options={"econv": 1e-6 * ions.nat},
        optimization_method="TN",
    )
    rho = opt.optimize_rho(guess_rho=rho0)
    energy = evaluator.Energy(rho=rho, ions=ions)
    print(f"Energy = {energy:.8f} Ha  ({energy / ions.nat:.8f} Ha/atom)")
    print("PP_list:", {k: Path(v).name for k, v in pseudo.PP_list.items()})
    return rho, energy

## 1. Pseudopotential already on disk

Classic workflow: map each element to a local file under `examples/DATA`.

In [2]:
ions, grid = make_al_system()

PP_list = {"Al": str(DATA / "al.lda.recpot")}
pseudo = LocalPseudo(grid=grid, ions=ions, PP_list=PP_list)

rho0 = guess_density(ions, grid)
rho_disk, E_disk = optimize_density(pseudo, ions, rho0)

setting key: Al -> /Users/michele/Documents/hackathon/DFTpy/examples/DATA/al.lda.recpot
Step    Energy(a.u.)            dE              dP              Nd      Nls     Time(s)         
0       -8.197204147412E+00     -8.197204E+00   1.195655E+00    1       1       1.256394E-02    
1       -8.440485696552E+00     -2.432815E-01   4.021914E-02    2       2       2.213693E-02    
2       -8.447003915870E+00     -6.518219E-03   2.489276E-03    6       3       3.767180E-02    
3       -8.447185358274E+00     -1.814424E-04   1.772435E-04    6       3       5.200505E-02    
4       -8.447197766527E+00     -1.240825E-05   1.486006E-05    7       2       6.559777E-02    
5       -8.447198659024E+00     -8.924968E-07   9.048198E-07    6       2       7.755995E-02    
6       -8.447198712157E+00     -5.313323E-08   7.434545E-08    5       2       8.848310E-02    
#### Density Optimization Converged ####
Chemical potential (a.u.): 0.287485897921542
Chemical potential (eV)  : 7.8228897449049155
Ener

## 2. Pseudopotential not on disk (OFPP download)

Minimal use: omit `PP_list`. DFTpy resolves missing species via OFPP (default families `OEPP` → `PGBRV0.2`), downloads into the cache on first use, and reuses the cache afterward.

In [3]:
ions, grid = make_al_system()

pseudo = LocalPseudo(grid=grid, ions=ions)

rho0 = guess_density(ions, grid)
rho_ofpp, E_ofpp = optimize_density(pseudo, ions, rho0)

setting key: Al -> /Users/michele/.cache/dftpy/ofpp/OEPP/Al_lda.oe01.recpot
Step    Energy(a.u.)            dE              dP              Nd      Nls     Time(s)         
0       -8.090977710718E+00     -8.090978E+00   7.877088E-01    1       1       9.108067E-03    
1       -8.273130665167E+00     -1.821530E-01   7.745403E-02    2       2       1.693106E-02    
2       -8.280424971016E+00     -7.294306E-03   7.026548E-03    6       2       2.844000E-02    
3       -8.281101144895E+00     -6.761739E-04   5.767546E-04    5       3       4.149103E-02    
4       -8.281133099107E+00     -3.195421E-05   5.322877E-05    4       2       5.034113E-02    
5       -8.281138667483E+00     -5.568375E-06   4.822344E-06    6       3       6.432509E-02    
6       -8.281138939070E+00     -2.715871E-07   4.070187E-07    4       2       7.319808E-02    
7       -8.281138995938E+00     -5.686817E-08   2.464409E-08    6       3       8.702683E-02    
#### Density Optimization Converged ####
Chemical p

## 3. Select a specific PP family

Pass `families` to restrict (or reorder) OFPP libraries. Examples: `OEPP`, `PGBRV0.2`, `HQLPP`, `OEPP:recpot`.

In [4]:
ions, grid = make_al_system()

pseudo = LocalPseudo(grid=grid, ions=ions, families=["HQLPP"])

rho0 = guess_density(ions, grid)
rho_fam, E_fam = optimize_density(pseudo, ions, rho0)

setting key: Al -> /Users/michele/.cache/dftpy/ofpp/HQLPP/al_lps.pbe.recpot
Step    Energy(a.u.)            dE              dP              Nd      Nls     Time(s)         
0       -7.609509082798E+00     -7.609509E+00   2.750577E+00    1       1       1.246905E-02    
1       -8.057560195695E+00     -4.480511E-01   1.366163E-01    4       3       2.759004E-02    
2       -8.068382454818E+00     -1.082226E-02   1.146609E-02    6       2       4.019904E-02    
3       -8.069031240178E+00     -6.487854E-04   8.808100E-04    4       3       5.209112E-02    
4       -8.069141224702E+00     -1.099845E-04   6.875028E-05    6       3       6.654310E-02    
5       -8.069145265237E+00     -4.040535E-06   4.507997E-06    4       2       7.644010E-02    
6       -8.069145686000E+00     -4.207636E-07   4.436604E-07    5       3       8.931017E-02    
7       -8.069145711762E+00     -2.576182E-08   4.789258E-08    4       2       9.874201E-02    
#### Density Optimization Converged ####
Chemical p

## 4. Config API (`[OFPP] auto = true`)

Same resolution through an in-memory config (or a `.ini` file). Manual `[PP]` entries override auto for that species.

In [5]:
ions, grid = make_al_system()

from dftpy.config import DefaultOption, OptionFormat
from dftpy.interface import ConfigParser

conf = DefaultOption()
conf["PATH"]["pppath"] = str(DATA)
conf["PATH"]["cellpath"] = str(DATA)
conf["CELL"]["cellfile"] = "fcc.vasp"
conf["CELL"]["elename"] = "Al"
conf["CELL"]["format"] = "vasp"
conf["GRID"]["nr"] = " ".join(map(str, grid.nr))
conf["EXC"]["xc"] = "LDA"
conf["KEDF"]["kedf"] = "TFvW"
conf["OFPP"]["auto"] = True
conf["OFPP"]["families"] = "OEPP PGBRV0.2"  # OEPP first, then PGBRV0.2
conf = OptionFormat(conf)

_, others = ConfigParser(conf, ions=ions, grid=grid)
pseudo = others["E_v_Evaluator"].PSEUDO

rho0 = guess_density(ions, grid)
rho_cfg, E_cfg = optimize_density(pseudo, ions, rho0)

The final grid size is  [20 20 20]
setting key: Al -> /Users/michele/Documents/hackathon/DFTpy/examples/DATA/Al_lda.oe01.recpot
Step    Energy(a.u.)            dE              dP              Nd      Nls     Time(s)         
0       -8.090977710718E+00     -8.090978E+00   7.877088E-01    1       1       8.453131E-03    
1       -8.273130665167E+00     -1.821530E-01   7.745403E-02    2       2       1.600099E-02    
2       -8.280424971016E+00     -7.294306E-03   7.026548E-03    6       2       3.130293E-02    
3       -8.281101144895E+00     -6.761739E-04   5.767546E-04    5       3       4.503703E-02    
4       -8.281133099107E+00     -3.195421E-05   5.322877E-05    4       2       5.413198E-02    
5       -8.281138667483E+00     -5.568375E-06   4.822344E-06    6       3       6.724095E-02    
6       -8.281138939070E+00     -2.715871E-07   4.070187E-07    4       2       7.644916E-02    
7       -8.281138995938E+00     -5.686817E-08   2.464409E-08    6       3       9.053326E-02    